# Lab 5.3 &mdash; Checkpointing: resume, approve, rewind, audit

**Time:** about 25 min &nbsp;|&nbsp; **Day 2 &middot; Module 5 &mdash; LangChain &amp; LangGraph**

### What you will do
- Write a checkpointer that saves the state after every node
- Stop a run in the middle, and resume it without doing work twice
- Pause before `open_incident` and wait for a person's approval
- Go back to an earlier checkpoint, change one value, and run again
- Read the saved states as an audit trail
- Do all of it again on real LangGraph, with `interrupt()` and a `thread_id`

> **How this lab works.** Fill in every `BLANK`, then run the **Self-check** cell under each
> section. It prints `[PASS]`, `[FAIL]` or `[TODO]` for each check. Graded cells never call the
> sandbox model, so your score does not depend on it. Cells marked **Run it for real** do call
> the model. If it is not reachable, they print how to fix it instead of crashing.

> **It builds on Lab 5.2.** One idea, saving the whole state after every node under a
> thread id, gives you all four: resume, approval, rewind and an audit trail.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, copy, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "aac-lab-5-3")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one check. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on."""
    try:
        return fn()
    except NameError:
        print("(a blank above is still unfilled -- fill it in, then run this cell again)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# The sandbox already has a model set up: nothing to install, no key to enter.
LLM_BASE_URL = os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
LLM_MODEL    = os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("No model is set up here. In a sandbox terminal run `env | grep LAB_LLM`.")
        print("If it prints nothing, tell your trainer. The graded cells still work.")
        return False
    return True

_llm = None
def get_llm(temperature: float = 0.0):
    """A LangChain chat model that talks to the sandbox model."""
    global _llm
    if _llm is None:
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                          api_key=LLM_API_KEY, temperature=temperature)
    return _llm

def ask(prompt: str, system: str | None = None) -> str:
    """One call to the model. Returns text, or an error string. Never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm().invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not set up -- the graded cells still work)")

In [ ]:
# ------------------------------------------------- AskOps: the same data and tools as Module 4
# Six runbooks and three open incidents. Nothing here is real, and nothing leaves this notebook.
RUNBOOKS = [{'id': 'RB-101',
  'title': 'Payments API returns 502 after deploy',
  'service': 'payments',
  'tags': ['deploy', 'rollback', '502', 'gateway'],
  'steps': ['Check the deploy pipeline for the last release',
            'Compare error rate before and after the release',
            'Roll back with the release tool if the error rate doubled',
            'Open an incident if rollback does not clear it']},
 {'id': 'RB-102',
  'title': 'Database connection pool exhausted',
  'service': 'payments',
  'tags': ['database', 'pool', 'timeout', 'connections'],
  'steps': ['Confirm the pool metric is at its maximum',
            'Find long-running queries and their owners',
            'Raise the pool size only as a temporary measure',
            'File a ticket for the query that held connections']},
 {'id': 'RB-201',
  'title': 'Login latency above 2 seconds',
  'service': 'auth',
  'tags': ['latency', 'login', 'cache', 'slow'],
  'steps': ['Check the token cache hit rate',
            'Warm the cache if a node restarted',
            'Scale the auth service if CPU is above 80 percent']},
 {'id': 'RB-202',
  'title': 'Certificate expiring within 7 days',
  'service': 'auth',
  'tags': ['tls', 'certificate', 'expiry'],
  'steps': ['List certificates expiring this week',
            'Request renewal from the PKI portal',
            'Deploy the renewed certificate and verify the chain']},
 {'id': 'RB-301',
  'title': 'Nightly batch job did not finish',
  'service': 'reporting',
  'tags': ['batch', 'job', 'timeout', 'retry'],
  'steps': ['Read the job log for the last completed step',
            'Re-run from the failed step, not from the start',
            'Tell report consumers the expected delay']},
 {'id': 'RB-302',
  'title': 'Disk usage above 90 percent on report nodes',
  'service': 'reporting',
  'tags': ['disk', 'storage', 'cleanup'],
  'steps': ['Find the largest directories',
            'Delete report archives older than 30 days',
            'Add a retention rule so it does not recur']}]

INCIDENTS_AT_START = [{'id': 'INC-9001',
  'title': 'Payments 502s after 14:00 release',
  'severity': 'high',
  'service': 'payments',
  'opened_at': '2026-09-14T14:07:00',
  'runbook_id': 'RB-101'},
 {'id': 'INC-9002',
  'title': 'Slow logins in the morning peak',
  'severity': 'medium',
  'service': 'auth',
  'opened_at': '2026-09-15T09:12:00',
  'runbook_id': 'RB-201'},
 {'id': 'INC-9003',
  'title': 'Batch report late for finance',
  'severity': 'low',
  'service': 'reporting',
  'opened_at': '2026-09-16T06:30:00',
  'runbook_id': None}]
INCIDENTS = copy.deepcopy(INCIDENTS_AT_START)

def reset_data():
    """Put the incident list back as it started. The checks call this, so they leave no trace."""
    INCIDENTS[:] = copy.deepcopy(INCIDENTS_AT_START)

# The four AskOps tools from labs/module-4-agent/agent.py, as plain functions.
def search_runbooks(query, service=None):
    words = {w for w in query.lower().split() if len(w) >= 3}
    hits = [{"id": r["id"], "title": r["title"]} for r in RUNBOOKS
            if service in (None, r["service"])
            and words & set(r["title"].lower().split() + r["tags"])]
    return json.dumps(hits[:3])

def get_runbook(runbook_id):
    for r in RUNBOOKS:
        if r["id"] == runbook_id:
            return json.dumps(r)
    return f"ERROR: no runbook {runbook_id}. Use search_runbooks first."

def list_incidents():
    return json.dumps(INCIDENTS)

def open_incident(title, severity, service, runbook_id=None):
    """The only tool that WRITES. It adds a record that other people see."""
    incident = {"id": f"INC-{9001 + len(INCIDENTS)}", "title": title,
                "severity": severity, "service": service, "runbook_id": runbook_id}
    INCIDENTS.insert(0, incident)
    return json.dumps(incident)

print(len(RUNBOOKS), "runbooks,", len(INCIDENTS), "open incidents")

In [ ]:
QUESTION_502  = "Payments returns 502 after deploy"            # INC-9001 already covers RB-101
QUESTION_DISK = "Disk on the report nodes is at 95 percent"    # RB-302, and no open incident

def fresh(question=QUESTION_DISK):
    """A new state for one question. Also puts the incident list back as it started."""
    reset_data()
    return {"question": question, "findings": [], "runbook_id": None,
            "already_open": False, "steps": 0, "answer": None}

def read_runbooks(s):
    """Node: find the best runbook for the question."""
    hits = json.loads(search_runbooks(s["question"]))
    rid = hits[0]["id"] if hits else None
    return {"findings": [f"runbook: {rid}"], "runbook_id": rid, "steps": s["steps"] + 1}

def read_incidents(s):
    """Node: is an incident for this runbook already open?"""
    covering = [i["id"] for i in json.loads(list_incidents())
                if s["runbook_id"] and i["runbook_id"] == s["runbook_id"]]
    return {"findings": [f"open incidents for it: {covering or 'none'}"],
            "already_open": bool(covering), "steps": s["steps"] + 1}

def service_of(runbook_id):
    return next((r["service"] for r in RUNBOOKS if r["id"] == runbook_id), "unknown")

def file_incident(s):
    """Node: the WRITE. Opens an incident, unless one is already open."""
    if s["already_open"]:
        return {"answer": "An open incident already covers this. Nothing opened.", "steps": s["steps"] + 1}
    opened = json.loads(open_incident(s["question"][:60], "medium", service_of(s["runbook_id"]),
                                      s["runbook_id"]))
    return {"answer": f"Opened {opened['id']}.", "steps": s["steps"] + 1}

def merge(state, update):
    """findings grow; every other key is replaced. The same reducers as Lab 5.2."""
    out = dict(state)
    for k, v in update.items():
        out[k] = list(state.get(k) or []) + list(v) if k == "findings" else v
    return out

END = "__end__"
NODES = {"read_runbooks": read_runbooks, "read_incidents": read_incidents, "file_incident": file_incident}
EDGES = {"read_runbooks": "read_incidents", "read_incidents": "file_incident", "file_incident": END}

## Concept

The AskOps graph in this lab has three nodes, and the last one **writes**:

```
read_runbooks --> read_incidents --> file_incident (opens an incident, unless one is already open)
```

After every node, the engine saves the whole state under a **thread id**. That one idea gives you:

| What you get | How the saved states give it to you |
|---|---|
| **Resume** | start again from the last saved state, not from the beginning |
| **Approval** | save, stop before the write, and continue when a person says yes |
| **Rewind** | load an earlier saved state, change one value, and run from there |
| **Audit trail** | the saved states are a record of what the agent knew at each step |

The audit trail is a **record**, not a summary written by the model. It is what the program really
held at each step.

## Section 1 &mdash; A checkpointer

It only ever adds entries, one list per thread. The real ones in LangGraph work the same way. They
differ mainly in where they save: memory, SQLite or a database.

In [ ]:
class Checkpointer:
    """A list of saved states for each thread. It only ever adds."""

    def __init__(self):
        self.threads: dict[str, list[dict]] = {}

    def put(self, thread: str, node: str, state: dict) -> None:
        """Save the state as it is AFTER `node` ran."""
        self.threads.setdefault(thread, []).append(
            {"seq": len(self.threads.get(thread, [])), "after": node,
             "state": json.loads(json.dumps(state))})     # a copy, not a reference

    def latest(self, thread: str) -> dict | None:
        """The most recent checkpoint, or None if the thread is new."""
        history = self.threads.get(thread, [])
        return history[-1] if history else None

    def at(self, thread: str, seq: int) -> dict | None:
        for cp in self.threads.get(thread, []):
            if cp["seq"] == seq:
                return cp
        return None

    def history(self, thread: str) -> list[dict]:
        return list(self.threads.get(thread, []))

In [ ]:
# --- Self-check: Section 1
def _cp():
    c = Checkpointer()
    c.put("t1", "read_runbooks", {"steps": 1, "findings": ["a"]})
    c.put("t1", "read_incidents", {"steps": 2, "findings": ["a", "b"]})
    return c

check("a new thread has no checkpoint", lambda: Checkpointer().latest("nope") is None)
check("latest returns the most recent", lambda: _cp().latest("t1")["after"] == "read_incidents")
check("history is in order and complete", lambda: [c["seq"] for c in _cp().history("t1")] == [0, 1])
check("an earlier checkpoint can still be read", lambda: _cp().at("t1", 0)["state"]["steps"] == 1)
def _snapshot_holds():
    c = Checkpointer()
    live = {"steps": 1, "findings": ["a"]}
    c.put("t1", "read_runbooks", live)
    live["findings"].append("changed later")
    return c.latest("t1")["state"]["findings"] == ["a"]
check("a checkpoint is a copy: later changes do not reach it", _snapshot_holds)

## Section 2 &mdash; Resume after a crash

The engine below saves after every node. It can also stop on purpose: `crash_after` pretends the
process died after a node, and `stop_before` pauses before a node. The test is simple: a resumed run
must **continue**, not start again. Here that matters, because starting again would open the
incident twice.

In [ ]:
def run_graph(state, thread, cp, max_steps=8, stop_before=None, crash_after=None):
    """Run the graph from the thread's last checkpoint, or from `state` if the thread is new.

    stop_before  -- pause before this node, and wait for approval
    crash_after  -- pretend the process died just after this node
    Returns (state, path, why), where why is done, paused, crashed or budget.
    """
    last = cp.latest(thread)
    current = "read_runbooks"
    if last:
        state, current = last["state"], last["state"].get("__next__", current)

    path = []
    while current != END:
        if state["steps"] >= max_steps:
            return merge(state, {"answer": "Stopped: step budget spent."}), path, "budget"
        if stop_before == current:
            cp.put(thread, "paused", {**state, "__next__": current})
            return state, path, "paused"
        path.append(current)
        state = merge(state, NODES[current](state))
        nxt = EDGES.get(current, END)
        cp.put(thread, current, {**state, "__next__": nxt})
        if crash_after == current:
            return state, path, "crashed"
        current = nxt
    return state, path, "done"

def resume(thread, cp, **kw):
    """Continue a thread from its last checkpoint."""
    last = cp.latest(thread)
    if last is None:
        raise ValueError("nothing to resume")
    return run_graph(last["state"], thread, cp, **kw)

In [ ]:
# --- Self-check: Section 2
def _crash_then_resume():
    cp = Checkpointer()
    s1, p1, why1 = run_graph(fresh(), "t", cp, crash_after="file_incident")
    before = len(INCIDENTS)
    # the process died after the write, but before anyone saw the answer
    s2, p2, why2 = resume("t", cp)
    return p1, why1, p2, why2, s2, before

def _crash_mid():
    cp = Checkpointer()
    run_graph(fresh(), "t", cp, crash_after="read_incidents")
    return resume("t", cp)

check("the run crashes where we said", lambda: _crash_then_resume()[1] == "crashed")
check("the resumed run finishes", lambda: _crash_mid()[2] == "done")
check("the resumed run does NOT repeat finished nodes",
      lambda: _crash_mid()[1] == ["file_incident"],
      "start from the saved state, not from a fresh one")
check("findings from before the crash survived", lambda: len(_crash_mid()[0]["findings"]) == 2)
def _no_double_write():
    p1, why1, p2, why2, s2, before = _crash_then_resume()
    return p2 == [] and before == 4 and len(INCIDENTS) == 4
check("after a crash just after the write, resuming does not write again", _no_double_write,
      "a resumed run that starts again opens a second incident")

## Section 3 &mdash; Pause for approval, and rewind

The same saved states, used in two more ways. `file_incident` writes a record that other people see,
so it is the node that waits for a person. **Rewind** loads an earlier state, changes one value and
runs again. It makes a new branch, and the original history stays as it was.

In [ ]:
def approve_and_continue(thread, cp):
    """A person said yes. Continue from where the run paused."""
    return resume(thread, cp)

def rewind(thread, cp, seq, changes: dict):
    """Go back to checkpoint `seq`, apply `changes`, and run again from there.

    Returns the new final state. The original history is left as it was:
    a rewind makes a new branch, it does not erase.
    """
    at = cp.at(thread, seq)
    if at is None:
        raise ValueError(f"no checkpoint {seq}")
    branch = Checkpointer()
    branch.threads[thread] = [c for c in cp.history(thread) if c["seq"] <= seq]
    branch.threads[thread][-1] = {**branch.threads[thread][-1], "state": {**at["state"], **changes}}
    return resume(thread, branch)[0]

In [ ]:
# --- Self-check: Section 3
def _paused():
    cp = Checkpointer()
    s, p, why = run_graph(fresh(), "t2", cp, stop_before="file_incident")
    return cp, s, p, why

check("the run pauses before the write", lambda: _paused()[3] == "paused")
check("it paused with both reads done", lambda: _paused()[2] == ["read_runbooks", "read_incidents"])
check("nothing was opened while it waits",
      lambda: (_paused(), len(INCIDENTS) == 3)[1],
      "the point of the pause is that the write has not happened yet")
check("approving continues to the end and opens one incident",
      lambda: approve_and_continue("t2", _paused()[0])[0]["answer"] == "Opened INC-9004.")

def _rewound():
    cp = Checkpointer()
    run_graph(fresh(QUESTION_502), "t3", cp)          # INC-9001 covers it: nothing is opened
    # A person says INC-9001 is a different problem. Go back to after read_incidents (seq 1).
    return cp, rewind("t3", cp, seq=1, changes={"already_open": False})

check("the first run opened nothing", lambda: len(_rewound()[0].history("t3")) == 3
      and "Nothing opened" in _rewound()[0].latest("t3")["state"]["answer"])
check("the rewound run, with one value changed, opens an incident",
      lambda: _rewound()[1]["answer"].startswith("Opened INC-"))
check("the original history is left as it was",
      lambda: len(_rewound()[0].history("t3")) == 3,
      "a rewind makes a new branch; it must not erase what really happened")

## Section 4 &mdash; The audit trail

The history already answers the question *what did the agent know, and when?* This cell prints it.

In [ ]:
def audit(thread, cp) -> str:
    """One line per checkpoint: after which node, what was known, and what came next."""
    rows = [f"{'seq':>4}  {'after':<16}{'steps':>6}{'findings':>10}  {'already_open':<14}next"]
    rows.append("-" * 76)
    for c in cp.history(thread):
        s = c["state"]
        rows.append(f"{c['seq']:>4}  {c['after']:<16}{s.get('steps', 0):>6}"
                    f"{len(s.get('findings', [])):>10}  {str(s.get('already_open')):<14}"
                    f"{s.get('__next__', '-')}")
    return "\n".join(rows)

try:
    _cpx = Checkpointer()
    run_graph(fresh(QUESTION_502), "audit-demo", _cpx)
    print(audit("audit-demo", _cpx))
except NameError:
    print("(finish the sections above, then run this cell again)")

In [ ]:
# --- Self-check: Section 4
def _audited():
    c = Checkpointer()
    run_graph(fresh(QUESTION_502), "a1", c)
    return c

check("one line per node, plus a header and a rule",
      lambda: len(audit("a1", _audited()).splitlines()) == 3 + 2)
check("the trail shows findings growing",
      lambda: "         1" in audit("a1", _audited()) and "         2" in audit("a1", _audited()))
check("the trail shows when already_open became true",
      lambda: audit("a1", _audited()).count("True") >= 2)
check("the trail comes from saved states, not from the model",
      lambda: all("state" in c for c in _audited().history("a1")),
      "this is why it is stronger evidence than asking the model what it did")

## Section 5 &mdash; The same, on real LangGraph

Now use LangGraph's own checkpointer. You need three things from the deck:

- `compile(checkpointer=InMemorySaver())` saves the state after every node.
- A `thread_id` in the config files each saved state under one conversation.
- `interrupt(value)` inside a node pauses the run and shows `value` to a person.
  `Command(resume=...)` continues it, and what you pass becomes the return value of `interrupt`.

One thing to remember: when the run continues, the node runs **again from its first line**. So never
put a write before the `interrupt`, or it happens twice.

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from operator import add
from langgraph.graph import StateGraph, START, END as LG_END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command

class TicketState(TypedDict):
    question: str
    findings: Annotated[list, add]
    runbook_id: str | None
    already_open: bool
    steps: int
    answer: str | None

def file_incident_approved(s):
    """Node: the WRITE, now behind a person's approval."""
    if s["already_open"]:
        return {"answer": "An open incident already covers this. Nothing opened.", "steps": s["steps"] + 1}
    draft = {"title": s["question"][:60], "severity": "medium",
             "service": service_of(s["runbook_id"]), "runbook_id": s["runbook_id"]}
    decision = interrupt(draft)
    if decision != "yes":
        return {"answer": "Not approved. Nothing opened.", "steps": s["steps"] + 1}
    opened = json.loads(open_incident(**draft))
    return {"answer": f"Opened {opened['id']}.", "steps": s["steps"] + 1}

def build_lg():
    g = StateGraph(TicketState)
    g.add_node("read_runbooks", read_runbooks)
    g.add_node("read_incidents", read_incidents)
    g.add_node("file_incident", file_incident_approved)
    g.add_edge(START, "read_runbooks")
    g.add_edge("read_runbooks", "read_incidents")
    g.add_edge("read_incidents", "file_incident")
    g.add_edge("file_incident", LG_END)
    return g.compile(checkpointer=InMemorySaver())

def cfg(thread):
    return {"configurable": {"thread_id": thread}}

In [ ]:
# Watch one run: it streams each node, then stops at the interrupt and shows the draft.
try:
    app = build_lg()
    for update in app.stream(fresh(QUESTION_DISK), cfg("eng-a"), stream_mode="updates"):
        for node, changes in update.items():
            print(f"{node:<16}", changes if node == "__interrupt__" else sorted(changes))
    print("\nwaiting before:", app.get_state(cfg("eng-a")).next)
    print("incidents now :", len(INCIDENTS))
    out = app.invoke(Command(resume="yes"), cfg("eng-a"))
    print("after 'yes'   :", out["answer"], "| incidents now:", len(INCIDENTS))
except NameError:
    print("(fill in the blank above, then run this cell again)")

In [ ]:
# --- Self-check: Section 5
def _lg_paused(thread="t-a"):
    app = build_lg()
    first = app.invoke(fresh(QUESTION_DISK), cfg(thread))
    return app, first

check("a question with no open incident pauses at the interrupt",
      lambda: "__interrupt__" in _lg_paused()[1])
check("the person sees the draft incident",
      lambda: _lg_paused()[1]["__interrupt__"][0].value["runbook_id"] == "RB-302")
check("the run waits before file_incident",
      lambda: _lg_paused()[0].get_state(cfg("t-a")).next == ("file_incident",))
check("nothing was opened while it waits", lambda: (_lg_paused(), len(INCIDENTS) == 3)[1])
def _lg_decide(answer):
    app, _ = _lg_paused("t-b")
    return app.invoke(Command(resume=answer), cfg("t-b"))["answer"]
check("'yes' opens exactly one incident",
      lambda: _lg_decide("yes") == "Opened INC-9004." and len(INCIDENTS) == 4)
check("'no' opens nothing", lambda: _lg_decide("no").startswith("Not approved") and len(INCIDENTS) == 3)
check("a covered question never pauses",
      lambda: "__interrupt__" not in build_lg().invoke(fresh(QUESTION_502), cfg("t-c")))
def _two_threads():
    app = build_lg()
    app.invoke(fresh(QUESTION_502), cfg("x"))       # finishes: INC-9001 covers it
    app.invoke(fresh(QUESTION_DISK), cfg("y"))      # pauses at the interrupt
    return app.get_state(cfg("x")).next == () and app.get_state(cfg("y")).next == ("file_incident",)
check("two threads keep separate states", _two_threads)
check("the saved history is the audit trail",
      lambda: len(list(_lg_paused("t-h")[0].get_state_history(cfg("t-h")))) >= 4)

In [ ]:
# Resume after a crash, on LangGraph. read_incidents fails once, as if the process died there.
try:
    calls = {"read_runbooks": 0}
    def counting_read_runbooks(s):
        calls["read_runbooks"] += 1
        return read_runbooks(s)
    failed = {"once": False}
    def flaky_read_incidents(s):
        if not failed["once"]:
            failed["once"] = True
            raise RuntimeError("the process died here")
        return read_incidents(s)

    g = StateGraph(TicketState)
    g.add_node("read_runbooks", counting_read_runbooks)
    g.add_node("read_incidents", flaky_read_incidents)
    g.add_edge(START, "read_runbooks")
    g.add_edge("read_runbooks", "read_incidents")
    g.add_edge("read_incidents", LG_END)
    app = g.compile(checkpointer=InMemorySaver())
    try:
        app.invoke(fresh(QUESTION_502), cfg("crash"))
    except RuntimeError as exc:
        print("crashed:", exc, "| waiting before:", app.get_state(cfg("crash")).next)
    out = app.invoke(None, cfg("crash"))       # None means: continue this thread
    print("resumed:", out["findings"])
    print("read_runbooks ran", calls["read_runbooks"], "time(s)")
except Exception as exc:
    print(f"<crash demo failed: {type(exc).__name__}: {exc}>")

## Run it for real

Ask the sandbox model to answer an auditor, using only the audit trail from Section 4. Notice what
it is doing: reading a record, not remembering a run.

In [ ]:
if llm_ready():
    try:
        cp = Checkpointer()
        run_graph(fresh(QUESTION_DISK), "real", cp)
        trail = audit("real", cp)
        answer = ask(
            "You are answering an auditor. Using ONLY this execution trail, say what the system knew "
            "when it opened an incident, and whether an incident was already open. If the trail does "
            "not support an answer, say so.\n\n" + trail
        )
        print(trail)
        print("\n--- the answer for the auditor ---\n" + answer.strip()[:600])
    except NameError:
        print("(finish the sections above, then run this cell again)")

### Read it

The model is summarising a **saved record**. Ask it the same question with no trail, and it writes
something just as confident that nobody can check.

So the honest answer to *can we explain what the agent did?* is: not from the model. From the
checkpoints.

In [ ]:
score()

## Your turn

1. `InMemorySaver` loses everything when the kernel restarts. The sandbox also has
   `langgraph-checkpoint-sqlite`. Change `build_lg` to use `SqliteSaver` with a file in `WORK`,
   pause a run, restart the kernel, and resume it.
2. The saved states hold the whole question and every finding. Which fields must never reach a
   checkpoint store under your team's data rules? Where would you remove them: in the node, the
   reducer, or the checkpointer?